# Dataset - Breast Cancer Wisconsin (Diagnostic) - UCI ML Repository

**Step 1 : Import all the required Libraries**

In [ ]:
# Core data handling
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Model persistence
import joblib
import json

# Dataset and train/test split
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Classification models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

# Evaluation metrics
from sklearn.metrics import (accuracy_score,roc_auc_score,precision_score,recall_score,f1_score,matthews_corrcoef,confusion_matrix,classification_report)

RANDOM_STATE = 27
print("Success : Imported Libraries")
print(f"Random State : {RANDOM_STATE}")

: 

**Step 2 : Load the dataset using Sklearn and print the dataset related data, Test and Train it**

In [ ]:
# Load Breast Cancer Wisconsin dataset (from UCI via sklearn)
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

print(f"Total number of of samples in the dataset are : {df.shape[0]}")
print(f"Total number of features excluding target: {df.shape[1] - 1}")
print(f"\nTarget distribution:")
print(df['target'].value_counts())
print(f"\n0 = Malignant, 1 = Benign")
df.head()

feature_names = list(data.feature_names)

train_df, test_df = train_test_split(df,test_size=0.20,random_state=RANDOM_STATE,stratify=df['target'])
X_train = train_df[feature_names]
y_train = train_df['target']
X_test = test_df[feature_names]
y_test = test_df['target']

# Streamlit app - Use this test data
test_df.to_csv('../test_data.csv', index=False)

print(f"Number of training samples after split: {len(X_train)}")
print(f"Number of test samples after split: {len(X_test)}")

**Step 3 : Train using Logistic Regression, decision Tree, K Nearest neighbours, Naive Bayes, random Forest**

In [ ]:
# Assignment : Define the five classification models
models = {
    # Used standardisation for logistic regression. This keeps all the on a comparable scale for linear classification model like logistic reression.
    "Logistic Regression": Pipeline([("scaler", StandardScaler()),("classifier", LogisticRegression(max_iter=1000,random_state=RANDOM_STATE))]),
    # Splitted based on feature thresholds for decision tree. Tree based model
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    # knn : Standardisation is done - Because it is calculated based on the nearest distance. For greater distance values, it can effect the computation. Hence standardisation.
    "kNN": Pipeline([("scaler", StandardScaler()),("classifier", KNeighborsClassifier(n_neighbors=5))]),
    # Used Gaussian Distribution.
    "Naive Bayes": GaussianNB(),
    # Using 150 trees for random forest
    "Random Forest": RandomForestClassifier(n_estimators=150,random_state=RANDOM_STATE)}

# Training all the above models
print("=" * 48)
print("5 MODELS TRAINING")
print("=" * 48)

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name:<25} : Training completed")

print("-" * 48)
print(f"All Details at a glance : ")
print(f"Training samples : {len(X_train)}")
print(f"Testing samples  : {len(X_test)}")
print(f"Number of models : {len(models)}")
print("=" * 48)

**Step 4 : Evaluate the accuracy,AUC,Precision,Recall,F1,MCC metrics for all the 5 models**

In [ ]:
# Evaluate all models using the six required classification metrics
results = []
for model_name, model in models.items():
    # Predictions for the test set
    y_pred = model.predict(X_test)
    # Probability of the positive class, required for ROC-AUC
    y_prob = model.predict_proba(X_test)[:, 1]
    # Calculate performance metrics
    model_results = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred)
    }
    results.append(model_results)
# Convert the results into a DataFrame
results_df = pd.DataFrame(results)
# Rounding off the metric values
metric_columns = ["Accuracy","AUC","Precision","Recall","F1","MCC"]
results_df[metric_columns] = results_df[metric_columns].round(4)
# Display the final comparison table
print("\n"+"="*70)
print("MODEL PERFORMANCE COMPARISON")
print("="*70)
print(results_df.to_string(index=False))
print("="*70)

**Step 5: Compute the best performing model overall**

In [ ]:
# Compare the above five models across all six evaluation metrics
metrics = ["Accuracy","AUC","Precision","Recall","F1","MCC"]
print("\n"+"="*50)
print("BEST MODEL FOR EACH METRIC")
print("="*50)
# Calculate count of each metric wins
metric_wins = {model: 0 for model in results_df["Model"]}
for metric in metrics:
    best_value = results_df[metric].max()
    best_models = results_df.loc[results_df[metric] == best_value,"Model"].tolist()
    # Print the best model and its value
    print(f"Best {metric:<10}: "f"{', '.join(best_models)} ({best_value:.4f})")
    # Count metric wins
    for model_name in best_models:
        metric_wins[model_name] += 1

# Metric vs number of wins
print("\n"+"-"*50)
print("Metric wise Wins")
print("-"*50)

for model_name, wins in metric_wins.items():
    print(f"{model_name:<25}: {wins}/{len(metrics)}")

# Determine the overall winner
highest_wins = max(metric_wins.values())
overall_winners = [model_name
    for model_name, wins in metric_wins.items()
    if wins == highest_wins]

# Final conclusion
print("\n"+"="*100)
print("Comparison and final conlcusion o the best Model : ")
print("="*100)

if len(overall_winners) == 1:
    winner = overall_winners[0]
    print(f"{winner} is the overall best-performing model, "
        f"with the best results in {highest_wins} "
        f"out of {len(metrics)} metrics."
    )
    highest_recall = results_df["Recall"].max()
    recall_models = results_df.loc[results_df["Recall"] == highest_recall,"Model"].tolist()
    print(f"{', '.join(recall_models)} achieved the highest"
        f"Recall of {highest_recall:.4f}.")
else:
    print(f"There is a tie between {', '.join(overall_winners)},"
        f"with {highest_wins} metric wins each.")

**Step 6 : Calculate the Confusion Matrix for the best model computed above**

In [ ]:
# using the overall winning model from the previous comparison
if len(overall_winners) == 1:
    best_model_name = overall_winners[0]
    best_model = models[best_model_name]

    # Generate predictions using the winning model
    y_pred = best_model.predict(X_test)
    # Calculate the confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    # Plot the confusion matrix
    plt.figure(figsize=(6,4))
    sns.heatmap(cm,annot=True,fmt="d",cmap="Greens",xticklabels=["Malignant", "Benign"],yticklabels=["Malignant", "Benign"])
    plt.xlabel("Predicted Class")
    plt.ylabel("Actual Class")
    plt.title(f"Confusion Matrix - {best_model_name}")
    plt.tight_layout()
    plt.show()
    # Classification report for the winning model
    print("\nClassification Report")
    print("-" * 60)
    print(f"Model: {best_model_name}")
    print("-" * 60)
    print(classification_report(y_test,y_pred,target_names=["Malignant", "Benign"]))
else:
    print("There is a tie between the overall winning models:")
    print(", ".join(overall_winners))
    print("Please select one model for the confusion matrix.")

**Step 7 : Save all the models to launch on Streamlit app**

In [ ]:
# Save the trained models
for name, model in models.items():
    filename = name.lower().replace(" ", "_") + ".pkl"
    joblib.dump(model, filename)
    print(f"Saved: {filename}")

# Save feature names for the Streamlit application
with open("feature_names.json", "w") as f:
    json.dump(feature_names, f)

# Save model evaluation results
with open("metrics.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nAll the 5 models and their supporting files have been saved successfully")
print("Now run 'streamlit run app.py' to launch the web app and verify.")